In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [3]:
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [4]:
%pwd

'c:\\Users\\91955\\OneDrive\\Desktop\\question_generator'

# DOCUMENT LOADING #


In [5]:
from langchain_community.document_loaders import PyPDFLoader

In [6]:
file_path = "C:/Users/91955/OneDrive/Desktop/question_generator/rag.pdf"
loader = PyPDFLoader(file_path)
data = loader.load()
print(data[0].page_content)

Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All
rights reserved. Draft of January 6, 2026.
CHAPTER
11
Information Retrieval and
Retrieval-Augmented Generation
On two occasions I have been asked,—“Pray, Mr. Babbage, if you put into
the machine wrong ﬁgures, will the right answers come out?” ... I am not able
rightly to apprehend the kind of confusion of ideas that could provoke such a
question. Babbage (1864)
People need to know things. So pretty much as soon as there were computers
we were asking them questions. By 1961 there was a system to answer questions
about American baseball statistics like “How many games did the Yankees play
in July?” (Green et al., 1961). Even ﬁctional computers in the 1970s like Deep
Thought, invented by Douglas Adams in The Hitchhiker’s Guide to the Galaxy ,
answered “the Ultimate Question Of Life, The Universe, and Everything”. 1 And
because so much knowledge is encoded in text, systems were answering questions
at h

In [7]:
len(data)

25

In [8]:
question_gen=""
for page in data:
    question_gen+=page.page_content


In [9]:
question_gen

'Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All\nrights reserved. Draft of January 6, 2026.\nCHAPTER\n11\nInformation Retrieval and\nRetrieval-Augmented Generation\nOn two occasions I have been asked,—“Pray, Mr. Babbage, if you put into\nthe machine wrong ﬁgures, will the right answers come out?” ... I am not able\nrightly to apprehend the kind of confusion of ideas that could provoke such a\nquestion. Babbage (1864)\nPeople need to know things. So pretty much as soon as there were computers\nwe were asking them questions. By 1961 there was a system to answer questions\nabout American baseball statistics like “How many games did the Yankees play\nin July?” (Green et al., 1961). Even ﬁctional computers in the 1970s like Deep\nThought, invented by Douglas Adams in The Hitchhiker’s Guide to the Galaxy ,\nanswered “the Ultimate Question Of Life, The Universe, and Everything”. 1 And\nbecause so much knowledge is encoded in text, systems were answeri

# CHUNKING #

In [10]:
from langchain_text_splitters import TokenTextSplitter

In [11]:
splitter_ques_gen=TokenTextSplitter(
    model_name="gpt-3.5-turbo",
    chunk_size=10000,
    chunk_overlap=200)

In [12]:
chunks_ques_gen=splitter_ques_gen.split_text(question_gen)

In [13]:
chunks_ques_gen

["Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All\nrights reserved. Draft of January 6, 2026.\nCHAPTER\n11\nInformation Retrieval and\nRetrieval-Augmented Generation\nOn two occasions I have been asked,—“Pray, Mr. Babbage, if you put into\nthe machine wrong ﬁgures, will the right answers come out?” ... I am not able\nrightly to apprehend the kind of confusion of ideas that could provoke such a\nquestion. Babbage (1864)\nPeople need to know things. So pretty much as soon as there were computers\nwe were asking them questions. By 1961 there was a system to answer questions\nabout American baseball statistics like “How many games did the Yankees play\nin July?” (Green et al., 1961). Even ﬁctional computers in the 1970s like Deep\nThought, invented by Douglas Adams in The Hitchhiker’s Guide to the Galaxy ,\nanswered “the Ultimate Question Of Life, The Universe, and Everything”. 1 And\nbecause so much knowledge is encoded in text, systems were answer

In [14]:
len(chunks_ques_gen)

2

In [15]:
type(chunks_ques_gen)

list

In [16]:
from langchain_core.documents import Document


In [17]:
documents_ques_gen=[Document(page_content=chunk) for chunk in chunks_ques_gen]

In [18]:
documents_ques_gen

[Document(page_content="Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All\nrights reserved. Draft of January 6, 2026.\nCHAPTER\n11\nInformation Retrieval and\nRetrieval-Augmented Generation\nOn two occasions I have been asked,—“Pray, Mr. Babbage, if you put into\nthe machine wrong ﬁgures, will the right answers come out?” ... I am not able\nrightly to apprehend the kind of confusion of ideas that could provoke such a\nquestion. Babbage (1864)\nPeople need to know things. So pretty much as soon as there were computers\nwe were asking them questions. By 1961 there was a system to answer questions\nabout American baseball statistics like “How many games did the Yankees play\nin July?” (Green et al., 1961). Even ﬁctional computers in the 1970s like Deep\nThought, invented by Douglas Adams in The Hitchhiker’s Guide to the Galaxy ,\nanswered “the Ultimate Question Of Life, The Universe, and Everything”. 1 And\nbecause so much knowledge is encoded in tex

In [19]:
type(documents_ques_gen[0])

langchain_core.documents.base.Document

In [20]:
splitter = TokenTextSplitter(model_name="gpt-3.5-turbo", chunk_size=500, chunk_overlap=50)

In [21]:
document_ans_gen=splitter.split_documents(documents_ques_gen)

In [22]:
document_ans_gen

[Document(page_content='Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All\nrights reserved. Draft of January 6, 2026.\nCHAPTER\n11\nInformation Retrieval and\nRetrieval-Augmented Generation\nOn two occasions I have been asked,—“Pray, Mr. Babbage, if you put into\nthe machine wrong ﬁgures, will the right answers come out?” ... I am not able\nrightly to apprehend the kind of confusion of ideas that could provoke such a\nquestion. Babbage (1864)\nPeople need to know things. So pretty much as soon as there were computers\nwe were asking them questions. By 1961 there was a system to answer questions\nabout American baseball statistics like “How many games did the Yankees play\nin July?” (Green et al., 1961). Even ﬁctional computers in the 1970s like Deep\nThought, invented by Douglas Adams in The Hitchhiker’s Guide to the Galaxy ,\nanswered “the Ultimate Question Of Life, The Universe, and Everything”. 1 And\nbecause so much knowledge is encoded in tex

In [23]:
len(document_ans_gen)

43

# GROQ API KEY #

In [24]:
from langchain_groq import ChatGroq

In [25]:
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

# PROMPT TEMPLATE #

In [26]:
prompt_template = """
You are an expert AI interviewer and question generation assistant.

Your task is to generate high-quality interview and exam preparation questions 
ONLY from the provided context.

Context:
----------------
{text}
----------------

Instructions:
- Generate clear and professional questions.
- Cover important concepts from the context.
- Include beginner, intermediate, and advanced questions.
- Include coding questions if the topic supports programming.
- Avoid duplicate questions.
- Do not generate questions outside the provided context.
- Preserve important technical information.

Output Format:
1.
2.
3.

QUESTIONS:
"""

In [27]:
from langchain_core.prompts import PromptTemplate

In [28]:
PROMPT = PromptTemplate.from_template(prompt_template)

In [29]:
from langchain.chains.summarize import load_summarize_chain

In [30]:
question_generation_chain = load_summarize_chain(
    llm,
    chain_type="map_reduce",
    map_prompt=PROMPT,
    combine_prompt=PROMPT
)

In [31]:
ques=question_generation_chain.run(documents_ques_gen)
print(ques)

C:\Users\91955\AppData\Local\Temp\ipykernel_13360\1223809743.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  ques=question_generation_chain.run(documents_ques_gen)
c:\Users\91955\.conda\envs\interview\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1170 > 1024). Running this sequence through the model will result in indexing errors


1. What is the primary goal of Information Retrieval (IR) systems, and how do they achieve this goal by representing documents and queries as vectors, specifically in the context of sparse and dense vector representations?

2. Explain the difference between sparse and dense vectors in IR, including their advantages and disadvantages, and provide examples of how they are used to represent documents and queries in various retrieval systems.

3. Describe the tf-idf weighting scheme and its role in computing term weights for documents and queries in the vector space model of IR, including the mathematical formulation and its significance in retrieval systems.

4. Write a Python function to calculate the tf-idf score for a given document and query, using the provided formulas, and explain how this function can be used in a simple IR system.

5. What is the concept of the inverted index in IR, and how is it used to efficiently find documents that contain words in the query, including its adv

In [32]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 687.83it/s]


In [33]:
from langchain.vectorstores import FAISS


In [34]:
vectorstore = FAISS.from_documents(document_ans_gen, embeddings)

In [35]:
ques

'1. What is the primary goal of Information Retrieval (IR) systems, and how do they achieve this goal by representing documents and queries as vectors, specifically in the context of sparse and dense vector representations?\n\n2. Explain the difference between sparse and dense vectors in IR, including their advantages and disadvantages, and provide examples of how they are used to represent documents and queries in various retrieval systems.\n\n3. Describe the tf-idf weighting scheme and its role in computing term weights for documents and queries in the vector space model of IR, including the mathematical formulation and its significance in retrieval systems.\n\n4. Write a Python function to calculate the tf-idf score for a given document and query, using the provided formulas, and explain how this function can be used in a simple IR system.\n\n5. What is the concept of the inverted index in IR, and how is it used to efficiently find documents that contain words in the query, includin

In [36]:
llm_ans_gen = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

In [37]:
ques_list=ques.split("\n")
ques_list

['1. What is the primary goal of Information Retrieval (IR) systems, and how do they achieve this goal by representing documents and queries as vectors, specifically in the context of sparse and dense vector representations?',
 '',
 '2. Explain the difference between sparse and dense vectors in IR, including their advantages and disadvantages, and provide examples of how they are used to represent documents and queries in various retrieval systems.',
 '',
 '3. Describe the tf-idf weighting scheme and its role in computing term weights for documents and queries in the vector space model of IR, including the mathematical formulation and its significance in retrieval systems.',
 '',
 '4. Write a Python function to calculate the tf-idf score for a given document and query, using the provided formulas, and explain how this function can be used in a simple IR system.',
 '',
 '5. What is the concept of the inverted index in IR, and how is it used to efficiently find documents that contain wor

In [38]:
from langchain.chains import RetrievalQA


In [39]:
answer_generation_chain = RetrievalQA.from_chain_type(
    llm_ans_gen,
    chain_type="stuff",
    retriever = vectorstore.as_retriever()
    
)

In [41]:
# Answer each question and save to a file
for question in ques_list:
    print("Question: ", question)

    answer = answer_generation_chain.run(question)

    print("Answer: ", answer)
    print("-----------------------------------------")

    # Save answer to file
    with open("answers.txt", "a") as f:
        f.write("Question: " + question + "\n")
        f.write("Answer: " + answer + "\n")
        f.write("-----------------------------------\n")

Question:  1. What is the primary goal of Information Retrieval (IR) systems, and how do they achieve this goal by representing documents and queries as vectors, specifically in the context of sparse and dense vector representations?


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01k8znzpk1f8f9fpw395d3jve9` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 97475, Requested 2795. Please try again in 3m53.28s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}